## Getting Started with Claude Managed Agents

### Installing Utilities and Libraries

In [17]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [18]:
import os
from pathlib import Path
from dotenv import load_dotenv

# override=True is required if this kernel already loaded empty values
for env_path in (Path(".env"), Path("Claude-Managed-Agents/.env")):
    if env_path.is_file():
        load_dotenv(env_path, override=True)
        break
else:
    raise FileNotFoundError("No .env file found. Expected Claude-Managed-Agents/.env")

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

if not claude_api_key:
    raise ValueError("CLAUDE_API_KEY is empty. Fill Claude-Managed-Agents/.env and re-run this cell.")

print(f"Loaded model={claude_model_name!r}, api_key_set={bool(claude_api_key)}")

Loaded model='claude-sonnet-5', api_key_set=True


### Creating the Anthropic Client

In [19]:
import anthropic

client = anthropic.Anthropic(api_key = claude_api_key)

### Create an Agent

In [20]:
agent = client.beta.agents.create(
    name="Demo-Managed-Agent",
    model=claude_model_name,
    system="You are a helpful AI Assistant.",
    tools=[
        {"type": "agent_toolset_20260401"},
    ],
)

print(f"Agent ID: {agent.id}, version: {agent.version}")

Agent ID: agent_01TZaVapTBVu5RHYR87dMDVi, version: 1


### Create an Environment

In [21]:
environment = client.beta.environments.create(
    name="quickstart-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)

print(f"Environment ID: {environment.id}")

Environment ID: env_011PE52MuRkozct7Vfgjua4C


### Start a Session

In [22]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    title="Quickstart session",
)

print(f"Session ID: {session.id}")

Session ID: sesn_01Gwe16ACLddbyGCZ3sSKMrF


### Start an Agent Loop

In [24]:
chat = True

while chat:
    user_query = input("Enter ""exit"" or your user query to continue")

    if user_query == "exit":
        chat = False
    else:
        with client.beta.sessions.events.stream(session.id) as stream:
            # Send the user message after the stream opens
            client.beta.sessions.events.send(
                session.id,
                events=[
                    {
                        "type": "user.message",
                        "content": [
                            {
                                "type": "text",
                                "text": user_query,
                            },
                        ],
                    },
                ],
            )

            # Process streaming events
            for event in stream:
                match event.type:
                    case "agent.message":
                        for block in event.content:
                            print(block.text, end="")
                    case "agent.tool_use":
                        print(f"\n[Using tool: {event.name}]")
                    case "session.status_idle":
                        print("\n\nAgent finished.")
                        break

No, I don't actually know you personally, Vaibhav. I don't have memory of past conversations (each chat starts fresh for me), and I don't have access to personal information, accounts, or data about you unless you share it with me directly in our conversation.

All I know right now is what you've told me here — your name is Vaibhav, and you asked about Navratri 2026. If you'd like, you can tell me more about yourself and I'll gladly use that context to help you better in this conversation!

Agent finished.
